In [1]:
# 네이버 블로그 요약 정보 수집하여 저장하기
#Step 0. 필요한 모듈과 라이브러리를 로딩하고 검색어를 입력 받습니다

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service  
from bs4 import BeautifulSoup
import time
import sys        # system 설정을 변경하기 위해 필요합니다
import math
import pandas  as pd    

In [2]:
#step 1. 사용자에게 필요한 정보를 입력받기
query_txt = input('1.크롤링할 키워드는 무엇입니까?: ')

try:
    cnt=int(input('2.수집할 데이터는 몇 건입니까?: ') )
except :
    cnt = 100

page_cnt = math.ceil(cnt / 30)

fc_name=input('3.csv 형태로 저장할 경로와 파일명을 확장자 포함해서 쓰세요(기본값:c:\\py_temp\\naver_blog.csv): ')
if fc_name=='':
    fc_name='c:\\py_temp\\naver_blog.csv'
    
fx_name=input('4.xlsx 형태로 저장할 경로와 파일명을 확장자 포함해서 쓰세요(기본값:c:\\py_temp\\naver_blog.xlsx): ')
if fx_name=='':
    fx_name='c:\\py_temp\\naver_blog.xlsx'

1.크롤링할 키워드는 무엇입니까?:  서진수 빅데이터
2.수집할 데이터는 몇 건입니까?:  15
3.csv 형태로 저장할 경로와 파일명을 확장자 포함해서 쓰세요(기본값:c:\py_temp\naver_blog.csv):  
4.xlsx 형태로 저장할 경로와 파일명을 확장자 포함해서 쓰세요(기본값:c:\py_temp\naver_blog.xlsx):  


In [3]:
#Step 2. 크롬 드라이버 설정 및 사이트 접속.
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

driver.get('http://www.naver.com')
driver.maximize_window( )
time.sleep(2)

In [4]:
#Step 3. 네이버 검색창에 입력 받은 검색어를 넣고 검색한 후 "블로그" 선택
element = driver.find_element(By.ID,"query")
element.send_keys(query_txt)
element.send_keys('\n')
time.sleep(5)

driver.find_element(By.LINK_TEXT,"블로그").click()

In [5]:
# Step 4. 저장 목록을 만들기
no2 = [ ]           # 게시글 번호 컬럼
title2 = [ ]        # 게시물 제목 컬럼
contents2 = [ ]     # 게시글 내용 컬럼
bdate2 = [ ]        # 작성 일자 컬럼
nick2 = [ ]         # 블로그 닉네임
url2 = [ ]          # 블로그 URL 주소

In [6]:
#Step 5. 현재 페이지의 내용을 저장 목록을 만든 후 목록에 있는 내용을 파일에 저장하기
# 자동 스크롤다운 함수
def scroll_down(driver):
  driver.execute_script("window.scrollTo(0,document.body.scrollHeight);")
  time.sleep(3)

i = 1
while (i <= page_cnt):
    scroll_down(driver) 
    print('%s 페이지 정보를 추출하고 있으니 잠시만 기다려 주세요~~^^' %i)
    i += 1

1 페이지 정보를 추출하고 있으니 잠시만 기다려 주세요~~^^


In [7]:
no = 1
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')

blog_list = soup.find('ul','lst_view').find_all('div','sds-comps-vertical-layout sds-comps-full-layout Rpk3YFQcZBMzoLEWz9U_')

In [8]:
len(blog_list)

30

In [13]:
for i in blog_list :
    print('\n')
    print('=' *100)
    
    no2.append(no)                            # 게시물 번호 리스트에 추가
    print('1.번호:',no)

    # 제목
    try:
        title = i.find('span','sds-comps-text sds-comps-text-ellipsis sds-comps-text-ellipsis-1 sds-comps-text-type-headline1 sds-comps-text-weight-sm').get_text()       # 게시물 제목
    except :
        title = '제목이 없습니다'
        title2.append(title)
    else :
        title2.append(title)                      # 게시물 제목 리스트에 추가
        print('2.제목:',title)

    # 요약내용
    try :
        contents = i.find('span','sds-comps-text sds-comps-text-type-body1 sds-comps-text-weight-sm').get_text()    # 게시물 내용
    except :
        contents = '본문 요약 내용이 없습니다'
        contents2.append(contents)                # 게시물 내용 리스트에 추가
        print('3.요약내용:',contents)
    else :
        contents2.append(contents)                # 게시물 내용 리스트에 추가
        print('3.요약내용:',contents)

    # 블로그 작성일자와 닉네임
    datenick = i.find('div','sds-comps-horizontal-layout').find_all('span','sds-comps-text')

    # 닉네임
    try:
        nick = datenick[0].get_text()   
    except :
        nick2 = '닉네임이 없습니다'
        nick2.append(nick)        
    else :    
        nick2.append(nick)
        print('4.닉네임:',nick)

    # 작성일자
    try:
        bdate = datenick[2].get_text()   
    except :
        bdate = '작성일자가 없습니다'
        bdate2.append(bdate)        
    else :    
        bdate2.append(bdate)
        
        print('5.작성일자:',bdate)

    # URL 주소
    try :
        url = i.find('div','sds-comps-vertical-layout').find('a')['href']
    except :
        url = 'URL 주소가 없습니다'
        url2.append(url)
    else :
        url2.append(url)
        print('6.URL:',url)

    time.sleep(1)

    print("\n")

    if no == cnt :
        break
        
    no += 1



1.번호: 1
2.제목: 빅데이터 전문가 서진수 강사 '4차산업혁명, 무엇을 대비해야하나' 특강
3.요약내용: 기업체나 공공기관 등 강연이 필요한 곳이라면 이제 스타강사랩과 함께해 보세요 인문학 역사학 동기부여 성공학 행복특강 등 수많은 분야의 전문가들을 맞춤섭외해 드립니다 평창농협 주최로 4차산업혁명 특강이 진행되었어요 강연자로는 빅데이터 전문가 서진수 강사가 함께해 주었는데요 '4차산업혁명, 무엇을 대비해야하나'라는 주제로 열띤 강연을 펼쳐주었어요... 
4.닉네임: 강사섭외는 스타강사랩
5.작성일자: 2025.07.16.
6.URL: https://blog.naver.com/rhtjrrlwkd




1.번호: 2
2.제목: 전문가에게 길을 묻다. 빅데이터 전문가편 서진수 대표
3.요약내용: 빅데이터 전문가편을 진행하였습니다. 각종 티비 프로그램과 인공지능과 관련된 18권의 책 출판, 국회에 4차 산업혁명 특별 자문위원, 세계인명사전에 과학자로 등재된 서진수 대표님과 함께 했습니다. 예전에 강연을 잘하셔서 국회의원 상을 받았다고 들었는데 역시나 아이들 눈맞춤으로 강연을 잘해주셨어요. 한시간 조금 넘게 진행하였는데, 아이들 모두... 
4.닉네임: 뉴턴의 과학실 과학전문학원
5.작성일자: 2022.08.06.
6.URL: https://blog.naver.com/n_jiyoung84




1.번호: 3
2.제목: [한국정보인재개발원_충북대학교] 줌(Zoom)을 이용한 비대면 교육 - R프로그래밍 활용 빅데이터 교육 후기(2025.08.04~07)
3.요약내용: R프로그램은 오픈소스 프로그램으로 통계/데이터 마이닝 및 그래프를 위한 언어입니다. 최근 빅데이터 분석을 목적으로 기업에서 많이 쓰이고 있으며 오픈 소스 프로그램이기 때문에 무료로 사용 가능하단 점이 큰 장점입니다. 이번 R프로그래밍 교육은 서진수 강사님께서 진행해주셨습니다.^^ 요즘 컴퓨터 언어에 대한 관심이 높아지고 있습니다! R프로그램을 활용한... 
4.닉네임: 한국정보인재개발원-빅데이터

In [16]:
# Step 6.출력 결과를 표(데이터 프레임) 형태로 만들기
# no2 = [ ]           # 게시글 번호 컬럼
# title2 = [ ]        # 게시물 제목 컬럼
# contents2 = [ ]     # 게시글 내용 컬럼
# bdate2 = [ ]        # 작성 일자 컬럼
# nick2 = [ ]         # 블로그 닉네임
# url2 = [ ]          # 블로그 URL 주소

naver_blog = pd.DataFrame()
naver_blog['번호']     = no2
naver_blog['제목']     = title2
naver_blog['요약내용'] = pd.Series(contents2)
naver_blog['작성일자'] = pd.Series(bdate2)
naver_blog['닉네임']   = pd.Series(nick2)
naver_blog['URL']     = pd.Series(url2)

# Step 7. 파일로 저장하기
# csv 형태로 저장하기
naver_blog.to_csv(fc_name,encoding="utf-8-sig",index=False)
print(" csv 파일 저장 경로: %s" %fc_name) 

# 엑셀 형태로 저장하기
naver_blog.to_excel(fx_name , index=False)
print(" xlsx 파일 저장 경로: %s" %fx_name) 

 csv 파일 저장 경로: c:\py_temp\naver_blog.csv
 xlsx 파일 저장 경로: c:\py_temp\naver_blog.xlsx
